#### Import libraries

In [17]:
!pip install xgboost

In [18]:
import pandas as pd
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import xgboost as xgb

In [19]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [20]:
df.shape

(10000, 11)

In [21]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

#### Data Cleaning

In [22]:
df = df.fillna(0)

In [23]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [24]:
# splitting data 60% train, 20% validation, 20% test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

print(len(df_train))
print(len(df_val))
print(len(df_test))

6000
2000
2000


In [ ]:
# seperate the target from features
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

# removing target columns from feature tables
df_train = df_train.drop(columns=['fuel_efficiency_mpg'])
df_val = df_val.drop(columns=['fuel_efficiency_mpg'])

In [26]:
# changing each row into a dict
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)


X_train = dv.fit_transform(train_dicts)

X_val = dv.transform(val_dicts)

print(dv.get_feature_names_out())

['acceleration' 'drivetrain=All-wheel drive'
 'drivetrain=Front-wheel drive' 'drivetrain=Rear-wheel drive'
 'engine_displacement' 'fuel_type=Diesel' 'fuel_type=Gasoline'
 'fuel_type=Hybrid' 'horsepower' 'model_year' 'num_cylinders' 'num_doors'
 'origin=Asia' 'origin=Europe' 'origin=USA' 'vehicle_weight']


#### Decision tree with max_depth = 1
Which original feature is used for splitting the data?

In [30]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

feature_number = dt.tree_.feature[0]
feature_name = dv.get_feature_names_out()[feature_number]
print('Feature used (after DictVectorizer):', feature_name)

original_column = feature_name.split('=')[0]
print('Original column:', original_column)

Feature used (after DictVectorizer): model_year
Original column: model_year


#### Random forest with n_estimators=10
What's the RMSE of this model on the validation data?

In [31]:
# random forest model train
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print('RMSE:', round(rmse,4))

RMSE: 1.8371
